# Merge Yoruba ASR Datasets → One Hugging Face Dataset

Combines four Yoruba speech datasets into one ASR corpus and pushes it to the Hub:

- `David-A-Amoo/naijavoices_dataset_85_hours_tts_best`  (audio stored in parquet, converted to proper audio here)
- `shunyalabs/yoruba-speech-dataset`
- `EYEDOL/naija-voices-yoruba-split_2-8`
- `michsethowusu/yoruba-speech-text-parallel`

**What this notebook does**
1. Inspects each source (streaming, no big downloads) so you can confirm column names and audio format
2. Loads every split of every source, auto-detects the text/audio columns, and converts parquet-stored waveforms into real audio
3. NFC-normalizes transcripts (important for Yoruba tone marks and underdots) and **removes any clip with fewer than 7 words**
4. Splits train/validation/test **by transcript hash** so the same sentence never lands in two splits
5. Computes total hours, then pushes to the Hub with a dataset card

**Before you run it**
- Some sources may be **gated** (NaijaVoices-derived data requires registration). Open each dataset page while logged in to the same HF account and accept the terms, and use a token that can read gated repos.
- **License:** NaijaVoices is CC BY-NC-SA 4.0 (non-commercial, share-alike). If any source carries it, the merged dataset must too. Check each source's dataset card and set `LICENSE_ID` in the config cell. Keep `PRIVATE = True` until you've confirmed redistribution is allowed.

## 1. Setup

In [1]:
# Do NOT add numpy here: upgrading it in a running session breaks the already-loaded copy (ImportError: _slice).
# Colab ships a compatible numpy already.
!pip install -q -U datasets huggingface_hub soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 17.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.2.0 which is incompatible.
tokenizers 0.23.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.2.0 which is incompatible.
gradio-client 2.7.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.2.0 which is incompatible.


In [2]:
from huggingface_hub import notebook_login

# Token needs READ access to gated repos and WRITE access to push
notebook_login()

## 2. Configuration

For each source, `text` / `audio` / `sr_col` can be left as `None` to auto-detect. If the inspection cell shows
the audio is stored as raw waveform arrays **without** a sampling-rate column, set `assumed_sr` for that source
(the notebook refuses to guess, because a wrong rate would silently ruin the audio).

In [3]:
import os, io, re, gc, shutil, hashlib, unicodedata, random
from collections import Counter

import numpy as np
import soundfile as sf
from datasets import (
    load_dataset, concatenate_datasets, Audio, Dataset, DatasetDict, Features, Value,
    get_dataset_config_names, get_dataset_split_names,
)
from huggingface_hub import list_repo_files, snapshot_download

HUB_REPO_ID = "Harbidel/yoruba-asr-merged"   # <-- change this
PRIVATE = True                                # keep True until licensing is confirmed
TARGET_SR = 16000
MIN_WORDS = 7                                 # clips with fewer words are removed
MAX_PER_TRANSCRIPT = None                     # e.g. 5 to cap how many speakers per identical sentence; None = keep all
FREE_HUB_CACHE = True                         # delete each source's raw parquet cache after it is loaded (saves a lot of disk)

LICENSE_ID = "cc-by-nc-sa-4.0"                # most restrictive license among sources -- verify each dataset card!
LICENSE_NOTE = (
    "This dataset inherits the most restrictive terms among its sources. Some sources derive from NaijaVoices "
    "(CC BY-NC-SA 4.0): non-commercial use only, attribution required, derivatives share alike. "
    "Check each source dataset card before reuse."
)

# Per-source options (all optional):
#   text / audio / sr_col : column names, None = auto-detect
#   assumed_sr            : sampling rate to assume when audio is raw waveforms with no rate column
#   data_files            : load specific files instead of letting `datasets` decide, e.g. "data/*.parquet"
#                           (use this if the repo has several formats and the wrong one gets picked up)
#   stream_filter         : stream the source and keep ONLY clips with >= MIN_WORDS words, so the rest is never
#                           written to disk. Use it for huge sources made mostly of very short clips.
SOURCES = [
    {"repo": "David-A-Amoo/naijavoices_dataset_85_hours_tts_best", "config": None, "text": None, "audio": None,
     "sr_col": None, "assumed_sr": None, "data_files": None, "stream_filter": False},
    {"repo": "shunyalabs/yoruba-speech-dataset", "config": None, "text": None, "audio": None,
     "sr_col": None, "assumed_sr": None, "data_files": None, "stream_filter": False},
    {"repo": "EYEDOL/naija-voices-yoruba-split_2-8", "config": None, "text": None, "audio": None,
     "sr_col": None, "assumed_sr": None, "data_files": None, "stream_filter": False},
    {"repo": "michsethowusu/yoruba-speech-text-parallel", "config": None, "text": None, "audio": None,
     "sr_col": None, "assumed_sr": None, "data_files": None, "stream_filter": True},   # ~1.6M tiny clips
]

## 3. Inspect each source (streaming: downloads almost nothing)

Read this output. For each source check: which column is the transcript, which is the audio, whether the audio
is a real `Audio` feature or something else (list of floats / bytes / dict), and whether a sampling-rate column exists.

In [4]:
def short(v, n=100):
    s = repr(v)
    return s if len(s) <= n else s[:n] + f"... ({len(s)} chars)"

for src in SOURCES:
    repo, cfg, dfiles = src["repo"], src["config"], src.get("data_files")
    print("=" * 70)
    print(repo)
    try:
        files = list_repo_files(repo, repo_type="dataset")
        ext = Counter(f.rsplit(".", 1)[-1].lower() if "." in f.rsplit("/", 1)[-1] else "(none)" for f in files)
        print(f"repo files: {len(files):,} | by extension: {dict(ext.most_common(8))}")
        print("first files:", files[:6])
    except Exception as e:
        print("could not list repo files:", type(e).__name__, e)
    try:
        if dfiles:
            print(f"using data_files={dfiles!r}")
            stream = load_dataset(repo, data_files=dfiles, split="train", streaming=True)
        else:
            configs = get_dataset_config_names(repo)
            print("configs:", configs)
            cfg_use = cfg or configs[0]
            splits = get_dataset_split_names(repo, cfg_use)
            print("splits:", splits)
            stream = load_dataset(repo, cfg_use, split=splits[0], streaming=True)
        print("features:", stream.features)
        row = next(iter(stream))
        for k, v in row.items():
            print(f"  {k}: {type(v).__name__} -> {short(v)}")
    except Exception as e:
        print("FAILED:", type(e).__name__, e)

David-A-Amoo/naijavoices_dataset_85_hours_tts_best
repo files: 3 | by extension: {'gitattributes': 1, 'md': 1, 'jsonl': 1}
first files: ['.gitattributes', 'README.md', 'metadata.jsonl']


README.md:   0%|          | 0.00/7.45k [00:00<?, ?B/s]

configs: ['default']
splits: ['train']
features: {'audio': Value('string'), 'text': Value('string'), 'duration_sec': Value('float64'), 'source_parquet': Value('string'), 'source_index': Value('int64')}
  audio: str -> './data/yoruba_wavs_best/yoruba_00000013.wav'
  text: str -> 'Mo n yan ipele aladun miiran ti awọn kuki chirún chocolate.'
  duration_sec: float -> 4.62
  source_parquet: str -> 'train-00000-of-00099.parquet'
  source_index: int -> 13
shunyalabs/yoruba-speech-dataset
repo files: 10 | by extension: {'parquet': 8, 'gitattributes': 1, 'md': 1}
first files: ['.gitattributes', 'README.md', 'data/test-00000-of-00002.parquet', 'data/test-00001-of-00002.parquet', 'data/train-00000-of-00005.parquet', 'data/train-00001-of-00005.parquet']


README.md:   0%|          | 0.00/588 [00:00<?, ?B/s]

configs: ['default']
splits: ['train', 'validation', 'test']
features: {'audio': Audio(sampling_rate=16000, decode=True, num_channels=None, stream_index=None), 'transcript': Value('string')}
  audio: AudioDecoder -> <datasets.features._torchcodec.AudioDecoder object at 0x7e08a40b7380>
  transcript: str -> 'dr.lee náà fi ẹ̀dùn ọkàn hàn lórí ìrìyìn pé àwọn ọmọ tó wà ní ilẹ̀ turkey ti kó kòkòrò ah5n1 èyí tó... (149 chars)
EYEDOL/naija-voices-yoruba-split_2-8
repo files: 12 | by extension: {'parquet': 10, 'gitattributes': 1, 'md': 1}
first files: ['.gitattributes', 'README.md', 'data/train-00000-of-00010.parquet', 'data/train-00001-of-00010.parquet', 'data/train-00002-of-00010.parquet', 'data/train-00003-of-00010.parquet']


...

configs: ['default']


Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

splits: ['train']


Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

features: {'audio': Audio(sampling_rate=None, decode=True, num_channels=None, stream_index=None), 'text': Value('string')}
  audio: AudioDecoder -> <datasets.features._torchcodec.AudioDecoder object at 0x7e08b9ecd810>
  text: str -> 'ti kó'


## 4. Helpers

`to_clean()` handles four audio layouts and always returns a dataset with proper 16 kHz `audio` + `text`:

| Layout in the source | What happens |
|---|---|
| Real `Audio` feature | cast to 16 kHz |
| Dict with `array` (+ optional `sampling_rate`) | re-encoded to real audio |
| Plain list of floats (+ sampling-rate column or `assumed_sr`) | re-encoded to real audio |
| Encoded bytes / file paths | cast to `Audio` |

In [5]:
TEXT_CANDIDATES  = ["text", "sentence", "transcription", "transcript", "normalized_text", "yoruba_text", "label"]
AUDIO_CANDIDATES = ["audio", "speech", "wav", "audio_data", "audio_array", "waveform", "array"]
SR_CANDIDATES    = ["sampling_rate", "sample_rate", "sr", "audio_sampling_rate"]


def pick(cols, candidates, explicit):
    if explicit:
        if explicit not in cols:
            raise ValueError(f"Column '{explicit}' not found. Available: {cols}")
        return explicit
    for c in candidates:
        if c in cols:
            return c
    return None


def find_audio_col(ds, explicit):
    if explicit:
        return pick(ds.column_names, [], explicit)
    for name, feat in ds.features.items():
        if isinstance(feat, Audio):
            return name
    return pick(ds.column_names, AUDIO_CANDIDATES, None)


def is_float_sequence(feat):
    inner = getattr(feat, "feature", None)
    if inner is None and isinstance(feat, list) and feat:
        inner = feat[0]
    return isinstance(inner, Value) and inner.dtype.startswith("float")


def waveform_to_dataset(ds, audio_col, text_col, sr_col, assumed_sr):
    """Re-encode raw waveform arrays stored in parquet into a proper Audio dataset (written to disk incrementally)."""
    cols = [audio_col, text_col] + ([sr_col] if sr_col else [])
    view = ds.select_columns(cols).with_format("numpy")

    # Fail early with a clear message if there is no way to know the sampling rate
    if len(view):
        first = view[0]
        a0 = first[audio_col]
        sr0 = a0.get("sampling_rate") if isinstance(a0, dict) else (first[sr_col] if sr_col else None)
        if sr0 is None and assumed_sr is None:
            raise ValueError(
                "Audio is stored as a raw waveform but no sampling rate was found. "
                "Set 'sr_col' or 'assumed_sr' for this source in SOURCES."
            )

    def gen():
        for ex in view:
            a = ex[audio_col]
            if isinstance(a, dict):
                arr, sr = a["array"], a.get("sampling_rate")
            else:
                arr, sr = a, (ex[sr_col] if sr_col else None)
            if sr is None:
                sr = assumed_sr
            if sr is None:
                raise ValueError(
                    "Audio is stored as a raw waveform but no sampling rate was found. "
                    "Set 'sr_col' or 'assumed_sr' for this source in SOURCES."
                )
            arr = np.asarray(arr, dtype=np.float32)
            if arr.ndim > 1:  # stereo -> mono
                arr = arr.mean(axis=0 if arr.shape[0] < arr.shape[-1] else -1)
            arr = np.clip(arr, -1.0, 1.0)
            # Encode to a real 16-bit WAV file in memory (works regardless of datasets/torch versions).
            # Stored at the source rate; Audio(sampling_rate=16000) resamples when the clip is decoded.
            buf = io.BytesIO()
            sf.write(buf, arr, int(sr), format="WAV", subtype="PCM_16")
            yield {"audio": {"bytes": buf.getvalue(), "path": None}, "text": str(ex[text_col])}

    # Emit a plain {bytes, path} struct, then cast to Audio (avoids Audio.encode_example, which needs torch/torchcodec)
    feats = Features({"audio": {"bytes": Value("binary"), "path": Value("string")}, "text": Value("string")})
    out = Dataset.from_generator(gen, features=feats)
    return out.cast_column("audio", Audio(sampling_rate=TARGET_SR))


def to_clean(ds, audio_col, text_col, src):
    feat = ds.features[audio_col]
    if isinstance(feat, Audio):
        kind = "audio"
    elif (isinstance(feat, dict) and "array" in feat) or is_float_sequence(feat):
        kind = "waveform"
    else:
        kind = "encoded"

    if kind == "waveform":
        sr_col = src.get("sr_col") or pick(ds.column_names, SR_CANDIDATES, None)
        return waveform_to_dataset(ds, audio_col, text_col, sr_col, src.get("assumed_sr")), kind

    out = ds.select_columns([audio_col, text_col])
    ren = {}
    if audio_col != "audio":
        ren[audio_col] = "audio"
    if text_col != "text":
        ren[text_col] = "text"
    if ren:
        out = out.rename_columns(ren)
    try:
        out = out.cast_column("audio", Audio(sampling_rate=TARGET_SR))
    except Exception as e:
        raise ValueError(f"Could not cast audio column '{audio_col}' (feature: {feat}) to Audio: {e}") from e
    return out, kind


def _stream_rows(repo, cfg, split, audio_col, text_col, min_words):
    """Stream a split and keep only clips with enough words, so unwanted audio is never written to disk."""
    s = load_dataset(repo, cfg, split=split, streaming=True)
    s = s.select_columns([audio_col, text_col]).cast_column(audio_col, Audio(decode=False))
    for ex in s:
        t = normalize_text(ex[text_col])
        if len(t.split()) < min_words:
            continue
        a = ex[audio_col]
        if not a or (a.get("bytes") is None and a.get("path") is None):
            continue
        yield {"audio": {"bytes": a.get("bytes"), "path": a.get("path")}, "text": t}


def stream_filtered_dataset(repo, cfg, split, src):
    probe = load_dataset(repo, cfg, split=split, streaming=True)
    feats = probe.features
    cols = list(feats) if feats else list(next(iter(probe)).keys())
    text_col = pick(cols, TEXT_CANDIDATES, src["text"])
    audio_col = (src["audio"]
                 or next((n for n, f in (feats or {}).items() if isinstance(f, Audio)), None)
                 or pick(cols, AUDIO_CANDIDATES, None))
    if text_col is None or audio_col is None:
        raise ValueError(f"{repo}: could not detect columns (columns: {cols}). Set 'text' / 'audio' in SOURCES.")
    gen_feats = Features({"audio": {"bytes": Value("binary"), "path": Value("string")}, "text": Value("string")})
    out = Dataset.from_generator(
        _stream_rows, features=gen_feats,
        gen_kwargs={"repo": repo, "cfg": cfg, "split": split, "audio_col": audio_col,
                    "text_col": text_col, "min_words": MIN_WORDS},
    )
    return out.cast_column("audio", Audio(sampling_rate=TARGET_SR)), text_col, audio_col


AUDIO_EXTS = (".wav", ".flac", ".mp3", ".ogg", ".opus", ".m4a")


def _norm_path(p):
    p = str(p).replace("\\", "/")
    while p.startswith("./"):
        p = p[2:]
    return p.lstrip("/")


def _file_rows(paths, texts):
    """Read audio files and emit their raw bytes, so the result no longer depends on the downloaded files."""
    for p, t in zip(paths, texts):
        if os.path.exists(p):
            with open(p, "rb") as f:
                yield {"audio": {"bytes": f.read(), "path": None}, "text": t}


def check_decode(clean):
    try:
        a = clean[0]["audio"]
        return True, f"{len(a['array']) / a['sampling_rate']:.1f}s @ {a['sampling_rate']} Hz | text: {clean[0]['text'][:50]!r}"
    except Exception as e:
        return False, e


def try_recover(src, ds, text_col, audio_col):
    """The table loaded but its audio does not decode (e.g. relative paths like ./data/x.wav). Look for the real audio in the repo."""
    repo = src["repo"]
    try:
        files = list_repo_files(repo, repo_type="dataset")
    except Exception as e:
        print("      recovery: could not list repo files:", e)
        return None

    # 1) the repo ships parquet files that were not the ones picked up
    parquet = [f for f in files if f.lower().endswith(".parquet")]
    if parquet:
        print(f"      recovery: repo has {len(parquet)} parquet file(s); loading those instead")
        ds2 = load_dataset(repo, data_files=parquet, split="train")
        t2 = pick(ds2.column_names, TEXT_CANDIDATES, src["text"])
        a2 = find_audio_col(ds2, src["audio"])
        return to_clean(ds2, a2, t2, src) if (t2 and a2) else None

    # 2) the table holds relative paths to individual audio files stored in the repo
    in_repo = {f for f in files if f.lower().endswith(AUDIO_EXTS)}
    if not in_repo:
        ext = Counter(f.rsplit(".", 1)[-1].lower() for f in files if "." in f.rsplit("/", 1)[-1])
        print(f"      recovery: no parquet or audio files in the repo (file types: {dict(ext.most_common(6))}). "
              "The audio may be zipped or hosted elsewhere.")
        return None
    rels = [_norm_path(p) for p in ds[audio_col]]
    texts = ["" if t is None else str(t) for t in ds[text_col]]
    keep = [i for i, r in enumerate(rels) if r in in_repo]
    if not keep:
        print("      recovery: the audio paths in the table do not match any audio file in the repo")
        return None
    print(f"      recovery: {len(keep):,} of {len(rels):,} rows point to audio files in the repo; downloading them")
    dirs = sorted({rels[i].rsplit("/", 1)[0] for i in keep if "/" in rels[i]})
    root = snapshot_download(repo, repo_type="dataset", allow_patterns=[d + "/*" for d in dirs] or ["*"])
    paths = [os.path.join(root, rels[i]) for i in keep]
    feats = Features({"audio": {"bytes": Value("binary"), "path": Value("string")}, "text": Value("string")})
    out = Dataset.from_generator(_file_rows, features=feats,
                                 gen_kwargs={"paths": paths, "texts": [texts[i] for i in keep]})
    return out.cast_column("audio", Audio(sampling_rate=TARGET_SR)), "repo-audio-files"


def _stream_rows(repo, cfg, split, audio_col, text_col, min_words):
    """Stream a split and keep only clips with enough words, so unwanted audio is never written to disk."""
    s = load_dataset(repo, cfg, split=split, streaming=True)
    s = s.select_columns([audio_col, text_col]).cast_column(audio_col, Audio(decode=False))
    for ex in s:
        t = normalize_text(ex[text_col])
        if len(t.split()) < min_words:
            continue
        a = ex[audio_col]
        if not a or (a.get("bytes") is None and a.get("path") is None):
            continue
        yield {"audio": {"bytes": a.get("bytes"), "path": a.get("path")}, "text": t}


def stream_filtered_dataset(repo, cfg, split, src):
    probe = load_dataset(repo, cfg, split=split, streaming=True)
    feats = probe.features
    cols = list(feats) if feats else list(next(iter(probe)).keys())
    text_col = pick(cols, TEXT_CANDIDATES, src["text"])
    audio_col = (src["audio"]
                 or next((n for n, f in (feats or {}).items() if isinstance(f, Audio)), None)
                 or pick(cols, AUDIO_CANDIDATES, None))
    if text_col is None or audio_col is None:
        raise ValueError(f"{repo}: could not detect columns (columns: {cols}). Set 'text' / 'audio' in SOURCES.")
    gen_feats = Features({"audio": {"bytes": Value("binary"), "path": Value("string")}, "text": Value("string")})
    out = Dataset.from_generator(
        _stream_rows, features=gen_feats,
        gen_kwargs={"repo": repo, "cfg": cfg, "split": split, "audio_col": audio_col,
                    "text_col": text_col, "min_words": MIN_WORDS},
    )
    return out.cast_column("audio", Audio(sampling_rate=TARGET_SR)), text_col, audio_col


def load_source(src):
    repo, cfg, dfiles = src["repo"], src["config"], src.get("data_files")
    if dfiles:
        split_names = ["train"]
    else:
        split_names = get_dataset_split_names(repo, cfg) if cfg else get_dataset_split_names(repo)
    extra = (f" | data_files={dfiles!r}" if dfiles else "") + (" | streaming filter" if src.get("stream_filter") else "")
    print(f"\n{repo}: splits = {split_names}{extra}")

    parts = []
    for split in split_names:
        ds = None
        if src.get("stream_filter"):
            clean, text_col, audio_col = stream_filtered_dataset(repo, cfg, split, src)
            kind = "stream-filtered"
        else:
            ds = load_dataset(repo, cfg, data_files=dfiles, split=split)
            text_col = pick(ds.column_names, TEXT_CANDIDATES, src["text"])
            audio_col = find_audio_col(ds, src["audio"])
            if text_col is None or audio_col is None:
                raise ValueError(
                    f"{repo}: could not detect columns (columns: {ds.column_names}). "
                    "Set 'text' / 'audio' for this source in SOURCES."
                )
            clean, kind = to_clean(ds, audio_col, text_col, src)

        ok, info = check_decode(clean) if len(clean) else (True, "no rows")
        if not ok and ds is not None and not dfiles:
            print(f"  [{split}] audio does not decode ({type(info).__name__}); looking for the real audio in the repo...")
            rec = try_recover(src, ds, text_col, audio_col)
            if rec is not None:
                clean, kind = rec
                ok, info = check_decode(clean)
        if not ok:
            raise RuntimeError(
                f"{repo}: audio could not be decoded ({type(info).__name__}: {str(info)[:200]}). "
                "Check the repo files printed in section 3, then set 'data_files' (e.g. a parquet glob) "
                "or 'audio' for this source in SOURCES."
            )

        print(f"  [{split}] {len(clean):,} rows | text='{text_col}' audio='{audio_col}' layout={kind}")
        if len(clean) == 0:
            print("      WARNING: no rows kept from this split")
            continue
        print(f"      sample check: {info}")

        # NOTE: no cast_column on text. Casting a text column rewrites the WHOLE dataset (audio included) to disk.
        # Rebuilding text/source with add_column is free and always yields a plain string type.
        texts = ["" if t is None else str(t) for t in clean["text"]]
        clean = clean.select_columns(["audio"]).add_column("text", texts)
        clean = clean.add_column("source", [f"{repo}:{split}"] * len(clean))
        parts.append(clean.select_columns(["audio", "text", "source"]))

    if not parts:
        raise ValueError(f"{repo}: no usable rows were loaded")
    return concatenate_datasets(parts)


def free_hub_cache(repo):
    path = os.path.expanduser(f"~/.cache/huggingface/hub/datasets--{repo.replace('/', '--')}")
    if os.path.isdir(path):
        shutil.rmtree(path, ignore_errors=True)
        print(f"  freed raw parquet cache for {repo}")


def disk_report(tag=""):
    total, used, free = shutil.disk_usage("/")
    print(f"[disk {tag}] free: {free / 1e9:.1f} GB of {total / 1e9:.1f} GB")


def normalize_text(t):
    t = unicodedata.normalize("NFC", t or "")
    return re.sub(r"\s+", " ", t).strip()

## 4b. Free disk after a failed or aborted run (skip on a fresh runtime)

Safe to run: it only deletes caches. Restart the session first if a load crashed, so nothing still points at the files being deleted.
Set `PURGE` to wipe the whole cache of a source that filled the disk.

In [6]:
import glob

disk_report("before cleanup")

# 1) raw parquet downloads of every source (not needed once the Arrow cache exists)
for src in SOURCES:
    free_hub_cache(src["repo"])

# 2) rewritten copies left behind by earlier failed runs
leftovers = glob.glob(os.path.expanduser("~/.cache/huggingface/datasets/**/cache-*.arrow"), recursive=True)
for f in leftovers:
    try:
        os.remove(f)
    except OSError:
        pass
print(f"removed {len(leftovers)} leftover cache-*.arrow files")

# 3) optionally wipe one source's whole Arrow cache
PURGE = []   # e.g. ["michsethowusu/yoruba-speech-text-parallel"]
for repo in PURGE:
    shutil.rmtree(os.path.expanduser(f"~/.cache/huggingface/datasets/{repo.replace('/', '___')}"), ignore_errors=True)
    print("purged", repo)

disk_report("after cleanup")

[disk before cleanup] free: 93.4 GB of 115.7 GB
  freed raw parquet cache for David-A-Amoo/naijavoices_dataset_85_hours_tts_best
  freed raw parquet cache for shunyalabs/yoruba-speech-dataset
  freed raw parquet cache for EYEDOL/naija-voices-yoruba-split_2-8
  freed raw parquet cache for michsethowusu/yoruba-speech-text-parallel
removed 0 leftover cache-*.arrow files
[disk after cleanup] free: 93.4 GB of 115.7 GB


## 5. Load and standardize every source

If a source fails here, the error message says exactly what to set in `SOURCES` (usually `text`, `audio`, `sr_col` or `assumed_sr`).
Gated-access errors mean you still need to accept that dataset's terms on its Hub page.

If one source fails, the others still load and are kept in memory; fix the failing source's entry in `SOURCES` and re-run this cell to retry only that one. If a table loads but its audio does not decode (e.g. relative paths like `./data/x.wav`), the loader looks in the repo for parquet files or the referenced audio files and uses those.\n\nA source flagged `stream_filter` is streamed and only clips with at least `MIN_WORDS` words are written to disk, so huge sources of tiny clips don't fill the disk.

In [8]:
SOURCES = [s for s in SOURCES if s["repo"] != "David-A-Amoo/naijavoices_dataset_85_hours_tts_best"]

In [9]:
disk_report("start")

# `loaded` survives re-runs of this cell: sources that already loaded are kept and not reloaded.
# To force a reload of one source:  loaded.pop("owner/name")
loaded = globals().get("loaded", {})
failed = {}

for src in SOURCES:
    repo = src["repo"]
    if repo in loaded:
        print(f"{repo}: already loaded ({len(loaded[repo]):,} rows), skipping")
        continue
    try:
        d = load_source(src)
    except Exception as e:
        failed[repo] = e
        print(f"\n!! {repo} FAILED: {type(e).__name__}: {str(e)[:400]}\n")
        continue
    loaded[repo] = d
    print(f"  -> {repo}: {len(d):,} rows total")
    if FREE_HUB_CACHE:
        free_hub_cache(repo)
    gc.collect()
    disk_report("after " + repo.split("/")[-1])

if failed:
    raise RuntimeError(
        "These sources failed: " + ", ".join(failed) + ". Fix their entries in SOURCES and re-run this cell; "
        "the sources that already loaded are kept."
    )

merged = concatenate_datasets([loaded[s["repo"]] for s in SOURCES])
print(f"\nMerged rows (before cleaning): {len(merged):,}")
merged

[disk start] free: 70.9 GB of 115.7 GB
shunyalabs/yoruba-speech-dataset: already loaded (3,548 rows), skipping
EYEDOL/naija-voices-yoruba-split_2-8: already loaded (20,738 rows), skipping
michsethowusu/yoruba-speech-text-parallel: already loaded (35,798 rows), skipping

Merged rows (before cleaning): 60,084


Dataset({
    features: ['audio', 'text', 'source'],
    num_rows: 60084
})

## 6. Normalize text and remove short transcripts

- NFC-normalizes and whitespace-collapses every transcript
- Removes clips with fewer than `MIN_WORDS` (7) whitespace-separated words
- Does **not** drop repeated sentences by default: several speakers reading the same prompt is valuable for ASR. Set `MAX_PER_TRANSCRIPT` in the config to cap it.

Text is swapped in as a new column, so the audio is not rewritten to disk.

In [10]:
src_base = [s.rsplit(":", 1)[0] for s in merged["source"]]
total_by_src = Counter(src_base)

norm = [normalize_text(t) for t in merged["text"]]
keep = [i for i, t in enumerate(norm) if len(t.split()) >= MIN_WORDS]
kept_after_words = Counter(src_base[i] for i in keep)

if MAX_PER_TRANSCRIPT:
    seen, capped = Counter(), []
    for i in keep:
        key = norm[i].lower()
        if seen[key] < MAX_PER_TRANSCRIPT:
            seen[key] += 1
            capped.append(i)
    keep = capped

kept_by_src = Counter(src_base[i] for i in keep)

print(f"{'source':<58}{'total':>9}{'<'+str(MIN_WORDS)+' words':>12}{'kept':>9}")
for s in total_by_src:
    short_dropped = total_by_src[s] - kept_after_words[s]
    print(f"{s:<58}{total_by_src[s]:>9,}{short_dropped:>12,}{kept_by_src[s]:>9,}")

merged = merged.remove_columns(["text"]).add_column("text", norm).select(keep)
merged = merged.select_columns(["audio", "text", "source"])
print(f"\nFinal rows: {len(merged):,}")

source                                                        total    <7 words     kept
shunyalabs/yoruba-speech-dataset                              3,548           1    3,547
EYEDOL/naija-voices-yoruba-split_2-8                         20,738       5,588   15,150
michsethowusu/yoruba-speech-text-parallel                    35,798           0   35,798

Final rows: 54,495


## 7. Split by transcript hash

Each unique sentence is hashed into train (~90%), validation (~5%) or test (~5%). All recordings of a sentence
land in the same split, so the test set never contains sentences the model trained on.

In [11]:
def bucket(text):
    h = int(hashlib.md5(text.lower().encode("utf-8")).hexdigest(), 16) % 100
    return "train" if h < 90 else ("validation" if h < 95 else "test")

idx = {"train": [], "validation": [], "test": []}
for i, t in enumerate(merged["text"]):
    idx[bucket(t)].append(i)

dataset_dict = DatasetDict({name: merged.select(ids) for name, ids in idx.items()})
for name, d in dataset_dict.items():
    print(f"{name:>10}: {len(d):,} clips")
dataset_dict

     train: 49,080 clips
validation: 2,738 clips
      test: 2,677 clips


DatasetDict({
    train: Dataset({
        features: ['audio', 'text', 'source'],
        num_rows: 49080
    })
    validation: Dataset({
        features: ['audio', 'text', 'source'],
        num_rows: 2738
    })
    test: Dataset({
        features: ['audio', 'text', 'source'],
        num_rows: 2677
    })
})

## 8. Sanity check (listen to a few clips)

In [12]:
import IPython.display as ipd

train = dataset_dict["train"]
for i in random.Random(0).sample(range(len(train)), 3):
    ex = train[i]
    print(f"[{ex['source']}] ({len(ex['text'].split())} words) {ex['text']}")
    display(ipd.Audio(ex["audio"]["array"], rate=ex["audio"]["sampling_rate"]))

[michsethowusu/yoruba-speech-text-parallel:train] (18 words) mo wólẹ̀ ní ẹsẹ̀ rẹ̀ bí ẹni tí ó kú. Ó sì fi ọwọ́ ọ̀tún rẹ̀ lé mi,


[michsethowusu/yoruba-speech-text-parallel:train] (12 words) lẹ́yìn tí Olúwa ti gba àwọn kan là láti ilẹ̀ Ejibiti wá,


[shunyalabs/yoruba-speech-dataset:test] (12 words) kọjá ìṣẹ̀lẹ̀ ọjọ́rú carpanedo kópa nínú ìdíje orísìí méjì níbi ìfigagbága olúbori


## 9. Total audio hours

Reads each clip's duration from its header (no full decode, no `.map()`, so no extra disk and little RAM).

In [13]:
def total_hours(ds, batch_size=500):
    ds = ds.cast_column("audio", Audio(decode=False))
    total, failed = 0.0, 0
    for i in range(0, len(ds), batch_size):
        for a in ds[i:i + batch_size]["audio"]:
            try:
                handle = io.BytesIO(a["bytes"]) if a.get("bytes") else a["path"]
                total += sf.info(handle).duration
            except Exception:
                failed += 1
    return total / 3600, failed

split_hours = {}
for name, d in dataset_dict.items():
    hours, failed = total_hours(d)
    split_hours[name] = hours
    note = f"  ({failed} clips unreadable)" if failed else ""
    print(f"{name:>10}: {hours:8.2f} hours  ({len(d):,} clips){note}")

total_hours_all = sum(split_hours.values())
print(f"{'TOTAL':>10}: {total_hours_all:8.2f} hours  ({sum(len(d) for d in dataset_dict.values()):,} clips)")

     train:    73.26 hours  (49,080 clips)
validation:     3.97 hours  (2,738 clips)
      test:     4.05 hours  (2,677 clips)
     TOTAL:    81.28 hours  (54,495 clips)


## 10. Push to the Hub

Starts **private**. Flip `PRIVATE` in the config only after you've confirmed every source license allows redistribution.

In [14]:
dataset_dict.push_to_hub(HUB_REPO_ID, private=PRIVATE, max_shard_size="500MB")
print(f"Pushed to https://huggingface.co/datasets/{HUB_REPO_ID}")

Uploading the dataset shards:   0%|          | 0/24 [00:00<?, ? shards/s]

Map:   0%|          | 0/2045 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmp8xs8xffk.parquet    :   3%|3         | 6.59MB /  210MB            

Validating                    :           |   0%            

Pushed to https://huggingface.co/datasets/Harbidel/yoruba-asr-merged


## 11. Dataset card

In [15]:
from huggingface_hub import HfApi

sources_md = "\n".join(
    f"- [{r}](https://huggingface.co/datasets/{r}): {kept_by_src[r]:,} of {total_by_src[r]:,} clips kept"
    for r in total_by_src
)
splits_md = "\n".join(
    f"- {name}: {len(d):,} clips ({split_hours[name]:.2f} hours)" for name, d in dataset_dict.items()
)

readme = f"""---
language:
- yo
license: {LICENSE_ID}
task_categories:
- automatic-speech-recognition
pretty_name: Merged Yoruba ASR Dataset
---

# {HUB_REPO_ID.split('/')[-1]}

A merged Yoruba speech-recognition dataset built from:

{sources_md}

## Processing

- Standardized to `audio` ({TARGET_SR // 1000} kHz) and `text` columns, plus a `source` column tracking origin
- Audio stored as raw waveform arrays in source parquet files was re-encoded into proper audio
- Transcripts Unicode NFC-normalized (important for Yoruba tone marks and underdots) and whitespace-collapsed
- Clips whose transcript has fewer than {MIN_WORDS} words were removed
- Splits are assigned by transcript hash (train ~90% / validation ~5% / test ~5%), so a sentence never appears in more than one split

## Splits

{splits_md}
- **Total: {sum(len(d) for d in dataset_dict.values()):,} clips ({total_hours_all:.2f} hours)**

## License

{LICENSE_NOTE}
"""

HfApi().upload_file(
    path_or_fileobj=readme.encode(),
    path_in_repo="README.md",
    repo_id=HUB_REPO_ID,
    repo_type="dataset",
)
print("README.md uploaded.")

README.md uploaded.
